## Wstęp do Uczenia Maszynowego

#### Laboratorium 04


### Zakres laboratorium

- walidacja krzyżowa i dobór hiperparametrów drzewa
- regresja liniowa
- regresja logistyczna
- interpretacja predykcji i współczynników modeli
- wprowadzenie do `Pipeline`


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
# Dane wykorzystywane w sekcji o walidacji krzyżowej.
pima_cv = pd.read_csv("../lab02/pima.csv")
X_cv = pima_cv.drop(columns="Outcome")
y_cv = pima_cv["Outcome"]
X_cv_train, X_cv_test, y_cv_train, y_cv_test = train_test_split(
    X_cv, y_cv, test_size=0.3, random_state=123
)


### 1. Dokończenie: walidacja krzyżowa i dobór hiperparametrów

Walidacja krzyżowa pozwala ocenić model wielokrotnie na różnych fragmentach zbioru treningowego. Wyniki CV służą do porównywania modeli i doboru hiperparametrów; zbiór testowy zachowujemy na końcową, jednorazową ocenę wybranego modelu.

### Kroswalidacja (*ang. cross-validation*) a Leave-One-Out

W $k$-krotnej walidacji krzyżowej dane treningowe dzielimy na $k$ części. Model uczymy $k$ razy, za każdym razem walidując go na innej części; końcowy wynik jest średnią wyników z foldów. Leave-One-Out to szczególny przypadek z $k=n$: każdy fold ma jedną obserwację walidacyjną. Wykorzystuje niemal wszystkie dane do uczenia, ale wymaga $n$ dopasowań modelu.


![](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)

Więcej: https://scikit-learn.org/stable/modules/cross_validation.html


---
##### *Zadanie 1*

Wymień zalety i wady kroswalidacji.


Zalety:


Wady:


---
##### *Zadanie 2*

Korzystając z przygotowanego wyżej zbioru treningowego `pima`, oszacuj jakość drzewa za pomocą 10-krotnej walidacji krzyżowej z miarą `accuracy`. Podaj średnią i odchylenie standardowe wyniku.


---
##### *Zadanie 3*

Na zbiorze treningowym wykonaj 100 powtórzeń 10-krotnej walidacji krzyżowej dla drzewa, używając miary `roc_auc`. Podaj średnią, odchylenie standardowe i narysuj rozkład wyników.


---
##### *Zadanie 4*

Dla danych `pima` zbadaj zachowanie hiperparametru `max_leaf_nodes` drzewa klasyfikacyjnego w zakresie od 2 do 30. Na danych treningowych i w 5-krotnej CV oblicz `accuracy`, `precision` i `roc_auc`. Przygotuj wizualizację, wybierz wartość według `roc_auc` z CV, a następnie raz oceń wybrany model na zbiorze testowym.

> **Uwaga.** Hiperparametr wybieramy wyłącznie na podstawie wyniku walidacji krzyżowej. Zbiór testowy służy tylko do końcowej oceny wybranego modelu.


### 2. Model regresji liniowej

Równanie modelu (w postaci macierzowej)

$$Y = X \beta + \epsilon$$

Gdzie:

- $Y$ – wektor zmiennej objaśnianej (wymiar $n \times 1$)

- $X$ – macierz cech / zmiennych objaśniających (wymiar $n \times (p+1)$, gdzie pierwsza kolumna to same jedynki dla wyrazu wolnego)

- $\beta$ – wektor nieznanych parametrów / współczynników regresji (wymiar $(p+1) \times 1$)

- $\epsilon$ – wektor składników losowych / błędów (wymiar $n \times 1$)

Kluczowe założenia modelu

- wartość oczekiwana błędu: $E(\epsilon) = 0$

- homoskedastyczność (stała wariancja błędów): $Var(\epsilon \vert{} X) = \sigma^2 I$ (gdzie $I$ to macierz jednostkowa)

- brak współliniowości: macierz $X$ ma pełny rząd kolumnowy, co oznacza, że macierz $X^T X$ jest odwracalna


Współczynniki $\beta$ wyznacza się, minimalizując sumę kwadratów reszt ($RSS$):

$$RSS = \epsilon^T \epsilon = (Y - X\beta)^T (Y - X\beta) \to \min_{\beta}.$$

Rozwiązanie tego problemu prowadzi do wzoru analitycznego na estymator współczynników $\hat{\beta}$:$$\hat{\beta} = (X^T X)^{-1} X^T Y.$$

Przewidywane wartości zmiennej objaśnianej ($\hat{Y}$) dla nowych danych oblicza się jako:$$\hat{Y} = X\hat{\beta}.$$

In [ ]:
# Wczytujemy zbiór danych do regresji liniowej
hills = pd.read_csv("hills.csv")

Zbiór danych dotyczący szkockich biegów górskich.
- dist – dystans trasy w milach,

- climb – suma przewyższenia w stópach,

- time – rekordowy czas (męski) w godzinach

> https://search.r-project.org/CRAN/refmans/DAAG/html/hills.html

-----
##### *Zadanie 5*

Dla danych `hills.csv`:

a) Narysuj wykres rozproszenia zmiennej *time* od *dist* oraz *time* od *climb*.

b) Oblicz współczynnik korelacji.


#### Budujemy model regresji liniowej dla pary zmiennych dist i time

-----
##### *Zadanie 6*

Na wykresie rozproszenia dla `dist` i `time` zaznacz wyestymowaną prostą z modelu liniowego.

-----
##### *Zadanie 7*

Zbuduj model regresji liniowej opisujący zależność *time* od *climb*, wyznacz współczynniki, narysuj wykres rozproszenia i zaznacz wyestymowaną prostą.

### 3. Model regresji logistycznej

W modelu regresji liniowej, predykcja jest ciągłą wartością (np. cena domu, wzrost), a model wygląda następująco:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n$$
$\hat{y}$ to przewidywana ciągła wartość.

$\beta_i$ to współczynniki (wagi) modelu.

$x_i$ to zmienne niezależne (cechy).

Ponieważ wartości prognozowane przez model regresji liniowej są rzeczywiste (od $-\infty$ do $+\infty$), a naszym celem jest klasyfikacja, stosujemy funkcję logistyczną jako przekształcenie.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
Gdzie $z$ to liniowa: $z = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n$.

Podstawiając predykcję liniową ($z$) do funkcji logistycznej, otrzymujemy model regresji logistycznej, który zwraca prawdopodobieństwo $P(Y=1|X)$:$$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_n x_n)}}$$

W regresji logistycznej istotne jest pojęcie logitu, który jest logarytmem naturalnym ze stosunku szans:$$\text{Logit} = \ln\left(\frac{P(Y=1|X)}{1 - P(Y=1|X)}\right)$$Zauważ, że logit jest równy liniowej kombinacji cech (wynikowi regresji liniowej):$$\ln\left(\frac{P(Y=1|X)}{1 - P(Y=1|X)}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n$$To jest powód, dla którego model nazywa się regresją logistyczną - ponieważ regresja jest wykonywana na logarytmie szans (logits).

-----
##### *Zadanie 8*

Przygotuj zbiór danych `SAheart.data` do dalszej pracy.


In [3]:
SAheart = pd.read_csv("SAheart.data")

-----
##### *Zadanie 9*

a) Dopasuj model regresji logistycznej do danych `SAheart.data`.

b) Oblicz prawdopodobieństwa przynależności do klasy 1 dla zbioru testowego.

c) Dokonaj predykcji dla obserwacji o wartościach `sbp = 127`, `age = 30`, `alcohol = 0`, `obesity = 25`, `typea = 49`, `famhist = 'Present'`, `tobacco = 0`, `ldl = 4.44`, `adiposity = 18`.

d) Oceń model na zbiorze testowym za pomocą `accuracy` i ROC AUC.


-----
##### *Zadanie 10*

Weźmy pod uwagę dane `earthquake.txt`, które dotyczą klasyfikacji wstrząsów (zmienna *popn*). Zmienne objaśniające to zmienne *body* i *surface* (zmienne sejsmologiczne).

a) Wykonaj wykres dla zmiennych *body* i *surface* z zaznaczeniem przynależności do klas.

b) Dopasuj model regresji logistycznej.

c) Zinterpretuj współczynniki modelu.

> **Uwaga.** Zbiór `earthquake` ma tylko 29 obserwacji. W tym przykładzie dopasowujemy model do całych danych wyłącznie po to, aby zilustrować granicę decyzyjną i interpretację współczynników — nie jest to ocena jakości modelu na nowych danych.


In [24]:
earthquake = pd.read_csv("earthquake.txt", sep=" ")

### 4. Wprowadzenie do `Pipeline`

W praktyce modelowanie zwykle obejmuje więcej niż samo dopasowanie klasyfikatora: na przykład uzupełnianie braków, kodowanie kategorii albo skalowanie zmiennych. Obiekt `Pipeline` łączy takie kroki w jeden estymator. Dzięki temu podczas `fit` transformacje są dopasowywane wyłącznie do zbioru treningowego, a następnie automatycznie stosowane do nowych danych.

W następnym laboratorium wykorzystamy `Pipeline` do pełnego przygotowania danych o różnych typach i z brakującymi wartościami.


#### Przykład: regresja logistyczna w `Pipeline`

Poniżej łączymy standaryzację i regresję logistyczną w jeden obiekt. Po wywołaniu `fit` skaler jest dopasowywany wyłącznie do danych treningowych, a następnie te same kroki są automatycznie stosowane przy `predict` i `predict_proba`.


In [ ]:
logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(C=np.inf, max_iter=300)),
])
logistic_pipeline.fit(X_train, y_train)

pipeline_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]
print("Accuracy pipeline'u na zbiorze testowym:",
      round(logistic_pipeline.score(X_test, y_test), 3))
print("Pierwsze prawdopodobieństwa:", np.round(pipeline_probabilities[:5], 3))
